# Implementing Logistic Regression from Scratch
## Samar Kamat

In this project, I will be referring to Andrew Ng's Machine Learning Specialization, and will be using a few mathematical formulae taught in the coursework.

### Logistic Regression

Logistic regression is a type of supervised learning in which a model is made to classify its predictions based on certain inputs. Logistic regression is particularly used for binary classification (predicting 0/1 or false/true). Unlike linear regression, which can predict a continuous value, logistic regression can predict from a finite set of classes (in binary classification, there are 2 classes). 

### Problem statement
For this notebook, I will be drawing inspiration from the last question of problem set 5, from Stanford's CS109 course. Instead of finding if people survived given certain conditions, I will be implementing a logistic regression model that predicts whether a passenger survived or not, given all the features (input variables). 

Specifically, I will be implementing the sigmoid function, L2 regularization, and the same logistic regression model in scikit-learn. 

The question: https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/problem12.html

### Dataset
The titanic.csv file contains data for 887 of the real Titanic passengers. The columns represent attributes about the person including whether they survived, their age, their passenger-class, their sex and the fare they paid. Each of the rows represents the data of one person.
The target (what we are trying to predict) is whether the person survived or not and the rest of the columns represent the features (input variables). 
I would like to thank Stanford, Kaggle and encyclopedia-titanica for the dataset.

**Data set**: https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import copy, math

In [2]:
df = pd.read_csv('https://web.stanford.edu/class/archive/cs/cs109/cs109.1166/stuff/titanic.csv') 
df.head()

,Survived,Pclass,Name,Sex,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


In [3]:
# preprocessing the data
df = df.drop('Name', axis=1)
df['Sex'] = df['Sex'].map({"male": 0, "female": 1})
df.head()

,Survived,Pclass,Sex,Age,Siblings/Spouses Aboard,Parents/Children Aboard,Fare
0,0,3,0,22.0,1,0,7.2500
1,1,1,1,38.0,1,0,71.2833
2,1,3,1,26.0,0,0,7.9250
3,1,1,1,35.0,1,0,53.1000
4,0,3,0,35.0,0,0,8.0500


In [4]:
X_df = df.iloc[:,1:7]
y_df = df.iloc[:, 0]

In [5]:
features = X_df.columns.tolist()
print(f"The features are: {features}")
target = y_df.name
print(f"Target is: {target}")

The features are: ['Pclass', 'Sex', 'Age', 'Siblings/Spouses Aboard', 'Parents/Children Aboard', 'Fare']
Target is: Survived


In [6]:
# Converting X to a matrix (2D numpy array), and y to a vector (1D numpy array)
X_train = X_df.to_numpy()
y_train = y_df.to_numpy()
print(f"Shape of X_train: {X_train.shape}")
print(X_train[:5])
print(f"Shape of y_train: {y_train.shape}")
print(y_train[:5])

Shape of X_train: (887, 6)
[[ 3.      0.     22.      1.      0.      7.25  ]
 [ 1.      1.     38.      1.      0.     71.2833]
 [ 3.      1.     26.      0.      0.      7.925 ]
 [ 1.      1.     35.      1.      0.     53.1   ]
 [ 3.      0.     35.      0.      0.      8.05  ]]
Shape of y_train: (887,)
[0 1 1 1 0]


### z-score normalization
Similar to linear regression, we normalize the feature values in logistic regression as well to more efficiently utilize gradient descent. Normalization is used to scale all of the features to similar ranges, so that the upcoming steps like gradient descent are performed much more quickly and efficiently.
z-score normalization ensures all of the features will have a mean of 0 and standard deviation of 1.

$$x^{(i)}_j = \dfrac{x^{(i)}_j - \mu_j}{\sigma_j} \tag{1}$$ 

Here, j is the feature, i is the example (row). $µ_j$ is the mean of all the examples of feature j and $\sigma_j$ is the standard deviation of feature j.
$$
\begin{align}
\mu_j &= \frac{1}{m} \sum_{i=0}^{m-1} x^{(i)}_j \tag{2}\\
\sigma^2_j &= \frac{1}{m} \sum_{i=0}^{m-1} (x^{(i)}_j - \mu_j)^2  \tag{3}
\end{align}
$$


In [7]:
# implementing z-score normalization
def z_norm(X_train):
    
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0)
    X_norm = (X_train - mu)/sigma
    
    return X_norm

### Sigmoid function
The purpose of the sigmoid function is to bring any continuous value into the range of 0 to 1 (inclusive). This helps bring numbers in a range that can be expressed in terms of probability quite easily. In binary classification, we can assume the output of the sigmoid function to be the probability of the output being 1. For instance, if 0 indicates the person didn't survive and 1 indicates the person survived, then 0.7 given by the sigmoid function would indicate there is a high probability the person survived. 

For logistic regression, the model is represented as:
$$ f_{\mathbf{w},b}(x) = g(\mathbf{w}\cdot \mathbf{x} + b) \tag{4}$$
Here, $g$ is the sigmoid function. The sigmoid function is defined as:
$$g(z) = \frac{1}{1+e^{-z}}\tag{5}$$
where z = $\mathbf{w}\cdot \mathbf{x} + b $

In [8]:
def sigmoid(z):

    z = np.clip(z, -500, 500)    
    g = 1/(1+np.exp(-z))

    return g

In [9]:
# testing the sigmoid function
val = 0
print(f"value: {val}, sigmoid({val}): {sigmoid(val)}")

value: 0, sigmoid(0): 0.5


### Cost function

The cost function helps us calculate the cost (how good our model is) given the current set of parameters. This helps us make sure that gradient descent is working properly, by making sure the cost reduces which iteration of gradient descent.

For logistic regression, the cost function is of the form:
$$ J(\mathbf{w},b) = \frac{1}{m}\sum_{i=0}^{m-1} \left[ loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) \right] \tag{6}$$

where m is the number of training set examples

The loss function mentioned in eq (3) is the cost for a single data point. Specifically it equates to:

$$loss(f_{\mathbf{w},b}(\mathbf{x}^{(i)}), y^{(i)}) = (-y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) \tag{7}$$

*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's prediction, while $y^{(i)}$, which is the actual label

*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = g(\mathbf{w} \cdot \mathbf{x^{(i)}} + b)$ where function $g$ is the sigmoid function.

### Cost function for Regularized Logistic Regression

We regularize the model because this ensures that the magnitude of the parameters is low, preventing overfitting of the model. This works because higher the parameter value, higher is it's squared value, resulting in higher costs (which is penalized).

For regularizing the logistic cost function (L2 regularization), we add a regularization term. Adding the regularization term:
$$\frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2 \tag{8}$$ 
to equation (6) above, our new cost function is:

$$J(\mathbf{w},b) = \frac{1}{m}  \sum_{i=0}^{m-1} \left[ -y^{(i)} \log\left(f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) - \left( 1 - y^{(i)}\right) \log \left( 1 - f_{\mathbf{w},b}\left( \mathbf{x}^{(i)} \right) \right) \right] + \frac{\lambda}{2m}  \sum_{j=0}^{n-1} w_j^2 \tag{9}$$
Note: parameter b is not recognized (it doesn't matter).

In [10]:
# implementing the cost function, with L2 regularization
def compute_cost(X_train, y_train, w, b, lambda_ = 1):

    m, n = X_train.shape
    total_cost = 0
    for i in range(m):
        z = np.dot(X_train[i], w) + b
        f_wb = sigmoid(z)
        total_cost += y_train[i]*(np.log(f_wb)) + (1-y_train[i])*(np.log(1-f_wb))

    total_cost = -1*(total_cost/m)

    reg = 0
    for j in range(n):
        reg += w[j]**2
    reg = (lambda_*reg)/(2*m)

    regularized_cost = total_cost + reg
    
    return regularized_cost

### Gradient for logistic regression

Part of reducing the cost, is to find the gradient. This means finding the direction to adjust each of the parameters, to reduce the overall cost during gradient descent. Therefore, we find the partial derivative of the cost function with respect to each of the parameters (the gradient for each).

$$
\frac{\partial J(\mathbf{w},b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)}) \tag{10}
$$
$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - \mathbf{y}^{(i)})x_{j}^{(i)} \tag{11}
$$

- here, m is the number of training examples in the dataset
- $f_{\mathbf{w},b}(x^{(i)})$ is the model's prediction, while $y^{(i)}$ is the actual label
- Note: this looks similar to the gradient calculation of linear regression, the difference is the definition of the $f_{\mathbf{w},b}(x)$ model.

### Regularized Gradient for logistic regression

To ensure be consistent with our cost function, we ensure that it's derivative also encorporates a regularized term. 

$$\frac{\partial J(\mathbf{w},b)}{\partial b} = \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{12}$$

$$\frac{\partial J(\mathbf{w},b)}{\partial w_j} = \left( \frac{1}{m}  \sum_{i=0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) x_j^{(i)} \right) + \frac{\lambda}{m} w_j  \quad\, \mbox{for $j=0...(n-1)$} \tag{13}$$

Notice that the bias is not regularized, so the gradient with respect to the bias is also not regularized.

In [11]:
# computing the regularized gradeint of logistic regression
def compute_gradient(X_train, y_train, w, b, lambda_ = 1):
    m, n = X_train.shape
    d_dw = np.zeros(w.shape)
    d_db = 0

    for i in range(m):
        z = np.dot(X_train[i], w) + b
        f_wb = sigmoid(z)
        diff = f_wb - y_train[i]

        for j in range(n):
            d_dw[j] += diff*X_train[i,j]
        d_db += diff

    d_dw = d_dw/m
    d_db = d_db/m

    for j in range(n):
        d_dw[j] += (lambda_*w[j])/m

    return d_dw, d_db

### Gradient Descent for logistic regression

The gradient descent algorithm is:
$$\begin{align*}& \text{repeat until convergence:} \; \lbrace \newline \; & b := b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b} \newline       \; & w_j := w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{14}  \; & \text{for j := 0..n-1}\newline & \rbrace\end{align*}$$

where, parameters $b$, $w_j$ are all updated simultaneously.
This appears to be the same as linear regression, the difference is in the actual gradient covered above (where the $f_{\mathbf{w},b}(x)$ model has a different definition).

In [12]:
# implementing gradient descent
def gradient_descent(X_train, y_train, w, b, alpha, iterations, compute_cost, compute_gradient, lambda_):
    
    cost_hist = []

    for i in range(iterations):

        if i < 100000:
            
            d_dw, d_db = compute_gradient(X_train, y_train, w, b, lambda_)
    
            w = w - (alpha*d_dw)
            b = b - (alpha*d_db)

            cost = compute_cost(X_train, y_train, w, b, lambda_)
            cost_hist.append(cost)

            if i % math.ceil(iterations / 10) == 0 or i == (iterations - 1):
                print(f"Iteration {i:4}: Cost {float(cost_hist[-1]):8.2f} ")

    return w, b

In [ ]:
m, n = X_train.shape
w_in = np.zeros((n,))
b_in = 0

iterations = 5000
alpha = 0.005
lambda_ = 0.5

X_train = z_norm(X_train)

w, b = gradient_descent(X_train, y_train, w_in, b_in, alpha, iterations, compute_cost, compute_gradient, lambda_)
print(f"Weights are: {w}")
print(f"Bias is: {b}")

Iteration    0: Cost     0.69 
Iteration  500: Cost     0.53 
Iteration 1000: Cost     0.49 
Iteration 1500: Cost     0.47 
Iteration 2000: Cost     0.46 
Iteration 2500: Cost     0.45 
Iteration 3000: Cost     0.45 
Iteration 3500: Cost     0.45 
Iteration 4000: Cost     0.44 


### Predict function

Assuming our output to be a continuous value from 0 to 1 (probability of survival), we want to make a prediction based on that, to essentially classify the continuous value into a yes/no or true/false answer. For that, we'll use the threshold of 0.5, meaning everything greater than or equal to 0.5 will be classified as 1, and everything below that is classified as 0.

To get a final prediction ($y^{(i)}=0$ or $y^{(i)}=1$) from the logistic regression model:

  if $f(x^{(i)}) >= 0.5$, predict $y^{(i)}=1$
  
  if $f(x^{(i)}) < 0.5$, predict $y^{(i)}=0$

In [ ]:
def predict(X_train, w, b):

    m, n = X_train.shape
    p = np.zeros(m)

    for i in range(m):
        z = np.dot(X_train[i], w) + b
        f_wb = sigmoid(z)

        if f_wb >= 0.5:
            p[i] = 1
        else:
            p[i] = 0

    return p

In [ ]:
# printing the accuracy of the model based on the training set
correct = 0
p = predict(X_train, w, b)
for i in range(m):
    if p[i] == y_train[i]:
        correct += 1

percentage = (correct/m)*100
print(f"Accuracy of model: {percentage:.3f}%")

### Implementing the same model in scikit-learn 
Scikit-learn is an open-source, commonly used toolkit, which contains the implementations of many machine learning algorithms.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
print(f"Scikit-learn model accuracy: {model.score(X_train, y_train)*100:.3f}%")

### Credits
This is my second project as part of my 'ML Projects from Scratch' collection. Hope you find this informative. I would love to thank Andrew Ng for his amazing Machine learning specialization, which has helped me grasp many machine learning concepts quite easily.